In [1]:
# Cell 0: multiprocessing / grpc fix
import os

# This tells gRPC to shut up and only show actual Errors
os.environ['GRPC_VERBOSITY'] = 'ERROR'

# This helps gRPC handle "forking" correctly in notebooks
os.environ['GRPC_ENABLE_FORK_SUPPORT'] = 'true'

In [2]:
# Cell 1: Imports + auto-find project root

import importlib
import os
import sys
from pathlib import Path
from IPython.display import HTML, display

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    try:
        for child in cwd.iterdir():
            if child.is_dir():
                candidates.append(child)
    except Exception:
        pass

    required_files = {
        "glucose_data_utils.py",
        "glucose_rules.py",
        "prompt_templates.py",
        "phia_agent.py",
    }

    for path in candidates:
        try:
            names = {p.name for p in path.iterdir()}
            if required_files.issubset(names):
                return path
        except Exception:
            pass

    raise FileNotFoundError("Could not find project root automatically.")

project_dir = find_project_root()
os.chdir(project_dir)

if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

print("project_dir:", project_dir)

from google import genai
from google.genai import types as genai_types

from glucose_data_utils import load_and_prepare_all
from glucose_rules import run_all_rules

import prompt_templates
import phia_agent

importlib.reload(prompt_templates)
importlib.reload(phia_agent)

from phia_agent import get_react_agent, QUESTION_PREFIX
from onetwo import ot
from onetwo.backends import google_genai_api

project_dir: /Users/esterpan/personal-health-insights-agent


In [3]:
#Cell 2:Set API key
tavily_api_key = "your api key"

In [4]:
# Cell 3: Setup LLM Backend

from google import genai
from google.genai import types as genai_types
from onetwo.backends import google_genai_api
import os

GOOGLE_API_KEY = "your api key"
MODEL_NAME = "models/gemini-2.5-pro"

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GEMINI_API_KEY"] = GOOGLE_API_KEY

client = genai.Client(api_key=GOOGLE_API_KEY)

llm_backend = google_genai_api.GoogleGenAIAPI(
    api_key=GOOGLE_API_KEY,
    vertexai=False,
    generate_model_name=MODEL_NAME,
    chat_model_name=MODEL_NAME,
    enable_model_verification=False,
    max_retries=8,
    initial_base_delay=2,
    max_base_delay=30,
    max_tokens=4096,
    temperature=0.2,
    http_options=genai_types.HttpOptions(api_version="v1beta"),
)

llm_backend.register()

print("GoogleGenAI backend registered:", MODEL_NAME)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


GoogleGenAI backend registered: models/gemini-2.5-pro


In [5]:
#Cell 4 — Load and prepare glucose data
data = load_and_prepare_all(
    cgm_path="sample_cgm.csv",      
    meals_path="sample_meals.csv",
    profile_path="sample_profile.csv",
    clean=True,
    use_overlap=True,
    overlap_anchor_shifts=(-10, 0, 10),
)

In [6]:
#Cell 5 — Unpack the tables
cgm_df = data["cgm_df"]
meals_df = data["meals_df"]
profile_df = data["profile_df"]
meal_features_df = data["meal_features_df"]
daily_df = data["daily_df"]

In [7]:
#Cell 6 — Run glucose rules
rule_outputs = run_all_rules(
    meal_features_df=meal_features_df,
    daily_df=daily_df,
    profile_df=profile_df,
)

meal_rule_df = rule_outputs["meal_rule_df"]
risk_df = rule_outputs["risk_df"]

In [8]:
#Cell 7 — Point to your few-shots
final_exemplar_paths = [
    "xt_few_shots/明确“不直接看原始 cgm_df”，先用高层表.ipynb",
    "xt_few_shots/什么时候该用 search.ipynb",
    "xt_few_shots/非诊断性建议生成.ipynb",
    "xt_few_shots/已知糖尿病用户的控制模式分类.ipynb",
    "xt_few_shots/一周血糖模式总结.ipynb",
    "xt_few_shots/单顿饭后的 spike 分析.ipynb",
    "xt_few_shots/健康用户风险提醒.ipynb",
]

for p in final_exemplar_paths:
    print(p, "FOUND" if os.path.exists(p) else "MISSING")

xt_few_shots/明确“不直接看原始 cgm_df”，先用高层表.ipynb FOUND
xt_few_shots/什么时候该用 search.ipynb FOUND
xt_few_shots/非诊断性建议生成.ipynb FOUND
xt_few_shots/已知糖尿病用户的控制模式分类.ipynb FOUND
xt_few_shots/一周血糖模式总结.ipynb FOUND
xt_few_shots/单顿饭后的 spike 分析.ipynb FOUND
xt_few_shots/健康用户风险提醒.ipynb FOUND


In [9]:
#cell 8 - create agent
agent = get_react_agent(
    cgm_df=cgm_df,
    meals_df=meals_df,
    meal_features_df=meal_features_df,
    daily_df=daily_df,
    meal_rule_df=meal_rule_df,
    risk_df=risk_df,
    profile_df=profile_df,
    example_files=final_exemplar_paths,
    tavily_api_key=tavily_api_key,
)

print("Glucose agent created successfully!")

Processing file: xt_few_shots/明确“不直接看原始 cgm_df”，先用高层表.ipynb
Processing file: xt_few_shots/什么时候该用 search.ipynb
Processing file: xt_few_shots/非诊断性建议生成.ipynb
Processing file: xt_few_shots/已知糖尿病用户的控制模式分类.ipynb
Processing file: xt_few_shots/一周血糖模式总结.ipynb
Processing file: xt_few_shots/单顿饭后的 spike 分析.ipynb
Processing file: xt_few_shots/健康用户风险提醒.ipynb
Glucose agent created successfully!


In [10]:
#cell 9 - one demo question
question = (
    "Based on my glucose data, what is the single most important issue I should pay attention to, "
    "what evidence supports it, why does it matter, and what is one practical next step I should try?"
)

full_question = QUESTION_PREFIX + question
print(full_question)

Use tools such as tool_code to execute Python code and search to find external relevant information as needed. Use the user's glucose-related tables whenever relevant. Prioritize meal_features_df, meal_rule_df, daily_df, and risk_df for most questions. Only use cgm_df for fine-grained raw time-series inspection. Use search only for background supplementation. Do not provide a formal medical diagnosis. Do not stop at generic summaries if actionable insights can be extracted from the data. Prioritize the most important 1 to 3 findings, explain why they matter, and suggest practical non-diagnostic next steps when appropriate. Take into account that questions may have typos or grammatical mistakes and try to reinterpret them before answering. Follow all instructions and the ReAct template carefully. Question: Based on my glucose data, what is the single most important issue I should pay attention to, what evidence supports it, why does it matter, and what is one practical next step I shoul

In [12]:
# Pre Cell 10: run the agent only when this smoke test passed
import time
import random

def smoke_test_25_pro(client, max_attempts=6):
    last_err = None
    for attempt in range(1, max_attempts + 1):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-pro",
                contents="Reply with exactly: OK"
            )
            text = getattr(resp, "text", None)
            if text and text.strip():
                print("Smoke test passed:", text)
                return True
            raise RuntimeError("Empty response from Gemini 2.5 Pro")
        except Exception as e:
            last_err = e
            wait_s = min(60, 2 ** attempt) + random.uniform(0, 2)
            print(f"[Smoke test {attempt}/{max_attempts}] failed: {e}")
            if attempt < max_attempts:
                print(f"Retrying in {wait_s:.1f}s...")
                time.sleep(wait_s)
    raise last_err

smoke_test_25_pro(client)

Smoke test passed: OK


True

In [13]:
# Cell 10: run the agent with retry

import time
import random

def run_agent_with_retry(agent, full_question, max_attempts=8):
    last_err = None

    for attempt in range(1, max_attempts + 1):
        print(f"[Attempt {attempt}/{max_attempts}] starting...", flush=True)
        try:
            final_answer, final_state = ot.run(
                agent(inputs=full_question, return_final_state=True)
            )
            print(f"[Attempt {attempt}] success", flush=True)
            return final_answer, final_state

        except Exception as e:
            last_err = e
            msg = str(e)
            print(f"[Attempt {attempt}] failed: {msg}", flush=True)

            retryable = (
                "503" in msg
                or "UNAVAILABLE" in msg
                or "high demand" in msg
                or "temporarily unavailable" in msg
            )

            if not retryable:
                raise

            wait_s = min(60, 2 ** attempt) + random.uniform(0, 2)
            print(f"[Attempt {attempt}] retrying in {wait_s:.1f}s...", flush=True)
            time.sleep(wait_s)

    raise last_err

final_answer, final_state = run_agent_with_retry(
    agent=agent,
    full_question=full_question,
    max_attempts=8,
)

print(final_answer)

[Attempt 1/8] starting...
[Attempt 1] success
This is an excellent question. It's wise to focus on the most impactful patterns first. Based on your glucose data, here is the single most important issue to pay attention to, the evidence behind it, why it matters, and a practical next step.

### 1. Most Important Insight
The most significant pattern in your data is the presence of **frequent and high post-meal glucose spikes**, especially after breakfast. These spikes often exceed the recommended range and show a delayed recovery.

### 2. Supporting Evidence from Your Data
- **High-Risk Meals**: Your `meal_rule_df` table shows that **75% of your meals are flagged as "High Risk"**. This is a very strong signal that post-meal responses are a primary concern.
- **Breakfast Spikes**: When looking at your `meal_features_df`, your breakfast meals consistently result in the highest glucose peaks. The average `peak_glucose` after breakfast is **210 mg/dL**, which is significantly above the targe

In [15]:
# Cell 11: display react trace + final answer

def escape_html(x):
    if x is None:
        return ""
    return (
        str(x)
        .replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
    )

def render_react_trace_html(state):
    blocks = []
    updates = getattr(state, "updates", [])

    for i, step in enumerate(updates, start=1):
        parts = [f"<div style='font-weight:700; margin-bottom:8px;'>Step {i}</div>"]

        thought = getattr(step, "thought", None)
        action = getattr(step, "action", None)
        observation = getattr(step, "observation", None)
        is_finished = getattr(step, "is_finished", False)

        if thought:
            parts.append(
                f"<div style='margin:6px 0;'><span style='font-weight:700;'>Thought:</span>"
                f"<div style='white-space:pre-wrap; margin-top:4px;'>{escape_html(thought)}</div></div>"
            )

        if action:
            parts.append(
                f"<div style='margin:6px 0;'><span style='font-weight:700;'>Act:</span>"
                f"<div style='white-space:pre-wrap; margin-top:4px;'>{escape_html(action)}</div></div>"
            )

        if observation:
            obs_text = str(observation)
            if len(obs_text) > 2000:
                obs_text = obs_text[:2000] + "\n...[truncated]..."
            parts.append(
                f"<div style='margin:6px 0;'><span style='font-weight:700;'>Observe:</span>"
                f"<div style='white-space:pre-wrap; margin-top:4px; background:#fafafa; padding:8px; border-radius:6px;'>{escape_html(obs_text)}</div></div>"
            )

        if is_finished:
            parts.append(
                "<div style='margin-top:8px; color:#0a7a0a; font-weight:700;'>Finished</div>"
            )

        block_html = (
            "<div style='border:1px solid #ddd; border-radius:8px; padding:12px; margin:12px 0;'>"
            + "".join(parts)
            + "</div>"
        )
        blocks.append(block_html)

    return "<div>" + "".join(blocks) + "</div>"

display(HTML("<h3>Agent Trace</h3>"))
display(HTML(render_react_trace_html(final_state)))

display(HTML("<h3>Final Answer</h3>"))
display(
    HTML(
        "<div style='border:1px solid #ddd; border-radius:8px; padding:14px; "
        "white-space:pre-wrap; line-height:1.6;'>"
        + escape_html(final_answer)
        + "</div>"
    )
)

In [16]:
#Cell 12 — Optional sanity check on high-level tables
print("cgm_df shape:", cgm_df.shape)
print("meals_df shape:", meals_df.shape)
print("meal_features_df shape:", meal_features_df.shape)
print("daily_df shape:", daily_df.shape)
print("meal_rule_df shape:", meal_rule_df.shape)
print("risk_df shape:", risk_df.shape)

print("\nMost common meal patterns:")
display(meal_rule_df["meal_pattern"].value_counts(dropna=False))

print("\nOverall risk labels:")
display(risk_df["overall_risk"].value_counts(dropna=False))

cgm_df shape: (7427, 3)
meals_df shape: (162, 4)
meal_features_df shape: (162, 23)
daily_df shape: (176, 14)
meal_rule_df shape: (162, 27)
risk_df shape: (30, 20)

Most common meal patterns:


meal_pattern
stable                66
hypoglycemia_risk     37
postprandial_spike    28
slow_recovery         22
persistently_high      6
high_variability       3
Name: count, dtype: int64


Overall risk labels:


overall_risk
low_risk                 15
attention_needed         11
needs_review              2
screening_recommended     1
relatively_stable         1
Name: count, dtype: int64